# Predictive Modeling Using Machine Learning
**Objectives:**
- Apply supervised learning algorithms: **Linear Regression**, **Decision Trees**, and **Random Forest**
- Train and test models for accuracy and generalization
- Visualize model performance using **Confusion Matrices** and **ROC Curves**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

sns.set_theme(style='whitegrid')

In [ ]:
# Load and prepare data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train models and evaluate
models = {
    'Linear Regression (Threshold 0.5)': LinearRegression(),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

fig_cm, axes_cm = plt.subplots(1, 3, figsize=(18, 5))
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
results = []

for idx, (name, model) in enumerate(models.items()):
    model.fit(X_train_scaled, y_train)
    if 'Linear Regression' in name:
        y_scores = np.clip(model.predict(X_test_scaled), 0, 1)
        y_pred = (y_scores >= 0.5).astype(int)
    else:
        y_pred = model.predict(X_test_scaled)
        y_scores = model.predict_proba(X_test_scaled)[:, 1]
        
    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_scores)
    results.append({'Model': name, 'Accuracy': acc, 'Precision': precision_score(y_test, y_pred), 'Recall': recall_score(y_test, y_pred), 'F1-Score': f1_score(y_test, y_pred), 'ROC-AUC': roc_auc})
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=data.target_names, yticklabels=data.target_names, ax=axes_cm[idx])
    axes_cm[idx].set_title(f'{name}\nAccuracy: {acc:.2%}')
    
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    ax_roc.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax_roc.plot([0, 1], [0, 1], 'k--', label='Random Guess')
ax_roc.set_title('ROC Curves Comparison')
ax_roc.legend()
plt.show()

pd.DataFrame(results)